# Phase 2 — Intent classification pipeline

Produces two tables:

- **`intent`**: `id, name, definition` — the fixed 12-intent taxonomy, as a lookup table.
- **`amazon_conversation`**: `conv_id, text, predicted_intent, confidence, intent_id` — one row per conversation, `intent_id` nullable.

Scope: the first 5,000 conversations from `amazonhelp_threads.csv` by `conv_id`. Model: `gpt-4o-mini` (cheaper than `gpt-4o`; already validated at 83.8% accuracy against the 80-row hand-labeled sample in `intent_classifier.ipynb` — no reason to pay for a bigger model on a 12-way classification task).

**Confidence rule** (as specified): the model estimates its own confidence (0-100) per classification. `intent_id` is only populated when confidence >= 75; below that, `intent_id` is left null. `predicted_intent` and `confidence` are still recorded even when null, so low-confidence cases can be audited rather than silently discarded.

**One tweet per conversation.** Only the customer's first turn is classified — see the reasoning in the next cell. Context length is a non-issue here: a single tweet is ~30-40 tokens, far under any model's limit; the constraint that matters is keeping each *batch* small enough that the model reliably returns complete, well-formed JSON for every item in it, not the model's context window.

## Why exactly one tweet (the first customer turn), not more

1. **Decision timing.** Intent classification exists to trigger real-time actions (route / draft / escalate) the moment a customer message arrives, before AmazonHelp has replied. A later customer turn only exists *after* an agent already responded, so it isn't available at the moment the decision is actually made — using it would leak information a production system won't have.
2. **One conversation, one label.** The taxonomy is one intent per conversation. If a customer's ask shifts mid-thread, blending turns would produce a fuzzy in-between label instead of a clean one; the first turn is unambiguous by construction.
3. **It's already sufficient.** These are tweets (median 130 characters), not multi-turn chat transcripts — the first message almost always states the full ask. There's real risk (leakage, ambiguity) in using later turns and little signal gained.

In [1]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found - check .env"

DATA = (Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent) / "Data"
MODEL = "gpt-4o-mini"
BATCH_SIZE = 50
MAX_WORKERS = 8
CONFIDENCE_THRESHOLD = 75
N_CONVERSATIONS = 5000

client = OpenAI()  # reads OPENAI_API_KEY from the environment

In [2]:
# Table 1: intent (id, name, definition). digital_content is explicitly scoped to software/content,
# not hardware - a fix made after intent_classifier.ipynb showed an Echo Dot hardware malfunction
# flip between product_defect and digital_content across two LLM passes.
INTENT_TABLE = pd.DataFrame([
    (1, "order_status", "Asking where an order is or for an ETA, with no complaint yet that anything is wrong."),
    (2, "delivery_problem", "The delivery itself failed: late, missing, misdelivered, or a rude/careless courier; 'delivered' but not actually received."),
    (3, "refund_or_return", "Wants money back or wants to send an item back, including A-to-z Guarantee claims."),
    (4, "cancellation_or_change", "Cancel or modify an order, or report that AmazonHelp/the seller cancelled one."),
    (5, "billing_or_payment", "Wrong, duplicate, or unexplained charge; payment declined; gift card can't be used; unauthorized charge."),
    (6, "product_defect", "The physical item or device itself is broken, wrong, or not as described (not a delivery failure)."),
    (7, "account_access", "Login, password, suspended/locked account, or phishing/fraud emails."),
    (8, "app_or_site_technical", "A site or app feature, or a checkout option, is broken or missing."),
    (9, "digital_content", "Kindle, Prime Video, Alexa/Echo software, or other digital-purchase content issues (not hardware malfunctions)."),
    (10, "subscription_membership", "Prime membership signup, benefits, renewal, or cancellation."),
    (11, "general_feedback_or_praise", "Venting or praise with no specific actionable request attached."),
    (12, "other", "Doesn't fit any of the above - policy/legal questions, contests, unrelated social posts."),
], columns=["id", "name", "definition"])

INTENT_TABLE.to_csv(DATA / "intent.csv", index=False)
NAME_TO_ID = dict(zip(INTENT_TABLE["name"], INTENT_TABLE["id"]))
INTENT_NAMES = set(INTENT_TABLE["name"])

print(f"wrote {len(INTENT_TABLE)} rows -> intent.csv")
INTENT_TABLE

wrote 12 rows -> intent.csv


,id,name,definition
0,1,order_status,"Asking where an order is or for an ETA, with n..."
1,2,delivery_problem,"The delivery itself failed: late, missing, mis..."
2,3,refund_or_return,Wants money back or wants to send an item back...
3,4,cancellation_or_change,"Cancel or modify an order, or report that Amaz..."
4,5,billing_or_payment,"Wrong, duplicate, or unexplained charge; payme..."
5,6,product_defect,"The physical item or device itself is broken, ..."
6,7,account_access,"Login, password, suspended/locked account, or ..."
7,8,app_or_site_technical,"A site or app feature, or a checkout option, i..."
8,9,digital_content,"Kindle, Prime Video, Alexa/Echo software, or o..."
9,10,subscription_membership,"Prime membership signup, benefits, renewal, or..."


In [3]:
threads = pd.read_csv(DATA / "amazonhelp_threads.csv", dtype={"in_response_to_tweet_id": "Int64"})

first_turn = (threads[threads["inbound"] == True]  # noqa: E712
              .sort_values(["conv_id", "turn_index"])
              .groupby("conv_id").first()
              .reset_index()[["conv_id", "text"]])

# First 5000 by conv_id = the 5000 chronologically OLDEST conversations (conv_id is numbered by
# each conversation's earliest tweet - see dataprep.ipynb). This is NOT a representative sample;
# it's whatever scope was asked for. Swap to .sample(N_CONVERSATIONS, random_state=0) instead if a
# representative subsample was actually intended.
scope = first_turn[first_turn["conv_id"] <= N_CONVERSATIONS].sort_values("conv_id").reset_index(drop=True)
print(f"classifying {len(scope):,} conversations (conv_id 1..{N_CONVERSATIONS})")

classifying 5,000 conversations (conv_id 1..5000)


In [4]:
TAXONOMY = "\n".join(f"{r.id}. {r.name} - {r.definition}" for r in INTENT_TABLE.itertuples())

SYSTEM_PROMPT = f"""You are labeling tweets sent by customers to @AmazonHelp with exactly one intent \
from this fixed taxonomy:

{TAXONOMY}

You will receive a JSON object {{"items": [{{"id": <int>, "text": <string>}}, ...]}}. For each item, \
pick the single best-fitting intent name from the list above and estimate your confidence (0-100) that \
this is genuinely correct given the definitions - low confidence is expected and fine when a message is \
ambiguous, sarcastic, off-topic, or doesn't clearly match any definition. Do not inflate confidence to \
avoid a low score.

Reply with ONLY a JSON object {{"results": [{{"id": <int>, "intent": <string>, "confidence": <0-100>}}, \
...]}}, one result per input item, same ids, using only the 12 intent names above verbatim."""

In [5]:
usage_lock_totals = {"prompt": 0, "completion": 0}

def classify_batch(items, retries=4):
    """items: list of (conv_id, text). Returns {conv_id: (intent_or_None, confidence_or_None)}.
    Never raises - a batch that fails every retry is marked as failed (both fields None) rather
    than crashing a 5000-row job over one bad batch."""
    payload = json.dumps({"items": [{"id": cid, "text": t} for cid, t in items]}, ensure_ascii=False)
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
            )
            if resp.usage:
                usage_lock_totals["prompt"] += resp.usage.prompt_tokens
                usage_lock_totals["completion"] += resp.usage.completion_tokens
            parsed = json.loads(resp.choices[0].message.content)
            out = {}
            for row in parsed["results"]:
                cid, intent, conf = int(row["id"]), row["intent"], float(row["confidence"])
                if intent not in INTENT_NAMES:
                    raise ValueError(f"unknown intent {intent!r}")
                out[cid] = (intent, conf)
            if set(out) != {cid for cid, _ in items}:
                raise ValueError("response ids don't match input ids")
            return out
        except Exception as e:  # noqa: BLE001 - retry on any transient parse/API error
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    print(f"  batch of {len(items)} failed after {retries} retries: {last_err!r}")
    return {cid: (None, None) for cid, _ in items}

def classify_all(df, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS):
    items = list(zip(df["conv_id"], df["text"]))
    batches = [items[i:i + batch_size] for i in range(0, len(items), batch_size)]
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(classify_batch, b) for b in batches]
        for done, fut in enumerate(as_completed(futures), start=1):
            results.update(fut.result())
            print(f"  batches done: {done}/{len(batches)}", end="\r")
    print()
    return results

In [6]:
results = classify_all(scope)

scope["predicted_intent"] = scope["conv_id"].map(lambda c: results[c][0])
scope["confidence"] = scope["conv_id"].map(lambda c: results[c][1])

# The confidence rule: map to intent_id only at >= threshold; otherwise null, even though we keep
# the raw predicted_intent/confidence around for auditing.
meets_threshold = scope["confidence"].notna() & (scope["confidence"] >= CONFIDENCE_THRESHOLD)
scope["intent_id"] = scope["predicted_intent"].map(NAME_TO_ID).where(meets_threshold)
scope["intent_id"] = scope["intent_id"].astype("Int64")

amazon_conversation = scope[["conv_id", "text", "predicted_intent", "confidence", "intent_id"]]
amazon_conversation.to_csv(DATA / "amazon_conversation.csv", index=False)
print(f"wrote {len(amazon_conversation):,} rows -> amazon_conversation.csv")

  batches done: 100/100
wrote 5,000 rows -> amazon_conversation.csv


In [7]:
n_failed = scope["predicted_intent"].isna().sum()
n_low_conf = ((~scope["predicted_intent"].isna()) & (scope["confidence"] < CONFIDENCE_THRESHOLD)).sum()
n_mapped = scope["intent_id"].notna().sum()

print(f"mapped (confidence >= {CONFIDENCE_THRESHOLD})      : {n_mapped:,} / {len(scope):,} "
      f"({n_mapped / len(scope):.1%})")
print(f"null - low confidence               : {n_low_conf:,} ({n_low_conf / len(scope):.1%})")
print(f"null - classification failed        : {n_failed:,} ({n_failed / len(scope):.1%})")
print(f"mean confidence (successful calls)  : {scope['confidence'].mean():.1f}")

print("\nmapped intent distribution:")
print(amazon_conversation.merge(INTENT_TABLE, left_on="intent_id", right_on="id")["name"]
      .value_counts().to_string())

print("\nsample of null (low-confidence) rows:")
low_conf_examples = scope[meets_threshold.eq(False) & scope["predicted_intent"].notna()]
for _, r in low_conf_examples.sample(min(10, len(low_conf_examples)), random_state=0).iterrows():
    print(f"  predicted={r['predicted_intent']:<26} conf={r['confidence']:>5.0f} | {r['text'][:70]}")

in_tok, out_tok = usage_lock_totals["prompt"], usage_lock_totals["completion"]
est_cost = in_tok / 1e6 * 0.15 + out_tok / 1e6 * 0.60  # gpt-4o-mini list pricing - verify current rate
print(f"\ntoken usage: {in_tok:,} prompt + {out_tok:,} completion -> est. ${est_cost:.3f} "
      f"(check current gpt-4o-mini pricing; this is a rough estimate)")

mapped (confidence >= 75)      : 3,175 / 5,000 (63.5%)
null - low confidence               : 1,825 (36.5%)
null - classification failed        : 0 (0.0%)
mean confidence (successful calls)  : 75.0

mapped intent distribution:
name
delivery_problem              1178
order_status                   343
refund_or_return               319
product_defect                 283
general_feedback_or_praise     264
billing_or_payment             218
app_or_site_technical          166
account_access                 114
cancellation_or_change         107
digital_content                105
subscription_membership         73
other                            5

sample of null (low-confidence) rows:
  predicted=app_or_site_technical      conf=   70 | .@115833 

When on the go...

Theres no way to access Alexa on actual 
  predicted=general_feedback_or_praise conf=   50 | when ya mom tells you that she signed up for amazon prime &amp; you ca
  predicted=refund_or_return           conf=   70 | @AmazonHelp 

## Caveats

- **These are the 5,000 chronologically oldest conversations, not a representative sample.** `conv_id` is numbered by each conversation's earliest tweet (see `dataprep.ipynb`), so this scope skews toward however AmazonHelp's Twitter support looked earliest in the dataset's time range. If the intent behind "first 5000" was a representative dev subsample rather than literally the oldest 5000, switch the slice in the data-loading cell to `.sample(N_CONVERSATIONS, random_state=0)`.
- **Confidence here is the model's self-reported estimate, not a calibrated probability.** A value of 75 from the model doesn't mean "75% of the time this is correct" in any measured sense — it's a heuristic signal. Treat the threshold as a tunable knob: use the golden set to check whether 75 is actually where accuracy falls off, once that set exists.
- **Some genuinely ambiguous cases (e.g. the `order_status` vs `delivery_problem` boundary found in `intent_classifier.ipynb`) may end up null rather than mis-routed** — the model tends to report lower confidence exactly on these boundary cases, which is arguably the correct behavior for this pipeline's purpose (hold uncertain cases rather than auto-route them wrong).